In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch

# Test it
print(f"Current Device: {torch.cuda.current_device()}")
print(f"Device Name: {torch.cuda.get_device_name(0)}") 
# Note: Because of the environment variable, GPU 1 now looks like '0' to PyTorch

Current Device: 0
Device Name: NVIDIA A100 80GB PCIe


# Based on Caustics + Caskade

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Ellipse
from matplotlib import colormaps
from IPython.display import HTML

from torch.nn.functional import conv2d, avg_pool2d
import torch
from torch import pi
from torch import where
from torch import Tensor

import caskade as ck
import caustics
from caustics import Module, forward, Param
from caustics import utils


from typing import Optional, Union, Annotated
from caskade import forward, Param
# from .base import Source, NameType
# from . import func

import copy
import h5py
import os
import scipy
import sncosmo
from PIL import Image
import galsim

from astropy.cosmology import WMAP9 as cosmo
from astropy.table import Table
from astropy.io import fits
from astropy import wcs

import matplotlib as mpl
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'serif'

# Camera kwargs
From Lenstronomy: <br>
https://docs.google.com/spreadsheets/d/1pMUB_OOZWwXON2dd5oP8PekhCT5MBBZJO1HV7IMZg4Y/edit?gid=0#gid=0
https://github.com/lenstronomy/lenstronomy/blob/main/lenstronomy/SimulationAPI/ObservationConfig/Roman.py

In [2]:
# Dictionary 
telescope_params = {
    'LSST': {
        'read_noise': 10.0,
        'pixel_scale': 0.2, 
        'ccd_gain': 2.3,
        'exposure_time': {'g': 15.0, 'r': 15.0, 'i': 15.0},
        'sky_brightness_mags': {'g': 22.26, 'r': 21.2, 'i': 20.48},
        'zero_points_mags': {'g': 28.30, 'r': 28.13, 'i': 27.79},
        'num_exposures': {'g': 200, 'r': 460, 'i': 460},
        'seeing': {'g':0.77, 'r':0.73, 'i':0.71}
    },
    'Roman': {
        'read_noise': 15.5,
        'pixel_scale': 0.11, 
        'ccd_gain': 1,
        'sky_brightness_mags': {'f062': 23.19, 'f087': 22.93, 'f106':22.99, 'f129':22.99, 'f158':23.10, 'f184':23.22, 'f146':22.03},
        'zero_points_mags': {'f062':26.6179, 'f087': 26.3023, 'f106': 26.3546, 'f129':26.3531, 'f158':26.3760, 'f184':25.9124, 'f146':27.5842},
        'seeing': {'f062': 0.058, 'f087':0.073, 'f106':0.087, 'f129': 0.106, 'f158':0.128, 'f184':0.146, 'f146':0.105},
        ## Wide on github:
        # 'exposure_time': {'f062':None, 'f087': None, 'f106': 107, 'f129': 107, 'f158': 107, 'f184': None, 'f146':None},
        # 'num_exposures':  {'f062':None, 'f087': None,   'f106': 6,   'f129': 6,   'f158': 6,   'f184': None, 'f146':None}
        ## Time Domain Wide on github
        'exposure_time': {'f062':60, 'f087': 85, 'f106': 95, 'f129': 152, 'f158': 294, 'f184': None, 'f146':None},
        'num_exposures':  {'f062':1, 'f087': 1,   'f106': 1,   'f129': 1,   'f158': 1,   'f184': None, 'f146':None}
    }
}

# Originally:  F062, F106, F184
#F184,  F158,  F129,  F146,  F106,  F087,   F062 ----> F106, F129,  F158
#Time domain wide: F158,  F129,  F106,  F087,   F062 
roman_bands = ['f106', 'f129',  'f158'] #['f087', 'f062', 'f158']


lsst_n_psf_g = 17
lsst_n_psf_r = 17
lsst_n_psf_i = 17
lsst_n_psf = 17


# SNcosmo lightcurve

In [3]:
def sncosmo_lightcurve(times, band, zero_point, z_source):
    if z_source > 0.915:
        z_source = 0.915
    model = sncosmo.Model(source='salt2-extended')
    model.set(z=z_source, t0=50, x1=0.1, c=-0.1)
    model.set_source_peakabsmag(-19.3, 'bessellb', 'ab')  # Calculate the correct x0 for the corresponding redshift
    x0 = model.get('x0')
    
    fluxes = model.bandflux(band, times, zp=zero_point, zpsys='ab')    # detector counts
    return times, fluxes

# Make function compatibe with caustics
def make_caustics_lightcurve_func(t_min, t_max, band, zp, z_source):
    times_np = np.linspace(t_min, t_max, 2000) 

    _, fluxes_np = sncosmo_lightcurve(times_np, band, zp, z_source) 
    times_tensor = torch.tensor(times_np, dtype=torch.float32)        #torch.Size([2000])
    fluxes_tensor = torch.tensor(fluxes_np, dtype=torch.float32)      #torch.Size([2000])
    def lc_func(t):
        return caustics.utils.interp1d(times_tensor, fluxes_tensor, t.reshape(-1) # Flatten for interpolation
                                      ).reshape(t.shape) # Return to image shape (undo flattening)
    return lc_func

# Make time variable point source class based on Caskade

Gaussian SN source based on: https://caskade.readthedocs.io/en/latest/notebooks/WorkedExample.html <br>

Galaxies are modeled as Sersic sources. Using Caustics Sersic class (no modifications). Source code: <br>
https://github.com/Ciela-Institute/caustics/blob/main/src/caustics/light/sersic.py <br>
and <br>
https://github.com/Ciela-Institute/caustics/blob/main/src/caustics/light/func/sersic.py

In [4]:
## From Caskade example
class TimeVaryingGaussian(caustics.Source):
    '''
    Based on Caskade's Gaussian source code --> in SN lightcurve example 
    q, phi, and sigma values taken and fixed from Caskade example 
        --> What values to use to model a Point Source?
    brightness returns Spacial spread of brightness (copied from Caskade) x Flux (function defined above)
    Multiband --> dictionary of flux functions --> for each band
    Setting simga to 0.03. Is this ok??
    '''
    def __init__(self, x0, y0, lc_funcs, q=1.0, phi=0.0, sigma=0.03, name = "SN",
                ):
        super().__init__(name=name)
        
        # Define parameters (copied from caustic Sersic sources class)
        self.x0 = caustics.Param("x0", x0, units="arcsec")
        self.y0 = caustics.Param("y0", y0, units="arcsec")
        self.q = caustics.Param("q", q, units="unitless")
        self.phi = caustics.Param("phi", phi, units="rad")
        self.sigma = caustics.Param("sigma", sigma, units="arcsec")
        self.lc_funcs = lc_funcs


    @forward
    def brightness(self, x, y, t, band, x0, y0, q, phi, sigma):
        x, y = x - x0, y - y0
        s, c = torch.sin(phi), torch.cos(phi)
        x, y = c * x - s * y, s * x + c * y
        r = (x**2 + (y * q) ** 2).sqrt()
        # Spacial shape based on Caskade example  -- how should we model a point source here ??
        spatial_shape = (-r ** 2 / sigma**2).exp() / (2 * np.pi * sigma**2).sqrt()  #?? --> osource code had sqrt but isnt that wrong??
        flux_at_t = self.lc_funcs[band](t)
        
        return flux_at_t * spatial_shape

# Build simulator - Caustics

In [9]:
class Singlelens(Module):
    def __init__(self, lens_mass, lens_light, host_light, sn_light, band_keys,
                 host_Ie_dict, lens_Ie_dict,
                 pixelscale, pixels_x, upsample_factor, psf_dict=None,
                 name: str = "sim"
                ):
        super().__init__(name)

        self.lens = lens_mass
        self.src_host = host_light
        self.src_sn = sn_light
        self.lens_light = lens_light

        self.host_Ie_dict = host_Ie_dict
        self.lens_Ie_dict = lens_Ie_dict
        self.band_keys = band_keys
        self.psf_dict = psf_dict
        self.upsample_factor = upsample_factor

        # Create the high-resolution grid
        theta_x, theta_y = caustics.utils.meshgrid(
            pixelscale / upsample_factor,  
            upsample_factor * pixels_x,    
            dtype=torch.float32,
        )
        # theta_x and theta_y --> lens plane coorindates
        self.theta_x = theta_x
        self.theta_y = theta_y

    @forward
    def run_simulator(self, times, psf_dict):
        # Get exact time delays at SN image coords
        beta_x_sn = self.src_sn.x0.value   # source plane coord of SN
        beta_y_sn = self.src_sn.y0.value   # source plane coord of SN
        try:
            theta_x_sn, theta_y_sn = self.lens.forward_raytrace(beta_x_sn, beta_y_sn) # get lens plane coords of SN ie coords of SN images
            n_images = theta_x_sn.shape[0]
        except (IndexError, ValueError):
            # If caustics finds no images, we catch it here
            n_images = 0

    
        if n_images == 2:
            sn_time_delays = self.lens.time_delay(theta_x_sn, theta_y_sn)
            sn_td_relative = torch.abs(sn_time_delays[0] - sn_time_delays[1])
            
            # SN image separations
            dx_sn = theta_x_sn[0] - theta_x_sn[1]
            dy_sn = theta_y_sn[0] - theta_y_sn[1]
            sep_sn_sn = torch.sqrt(dx_sn**2 + dy_sn**2)
            
            # SN to host center (host is in source plane so ray-trace host center to lens plane)
           # SN to lens center - compute first before try/except
            lens_x = self.lens.x0.value
            lens_y = self.lens.y0.value
            sep_sn0_lens = torch.sqrt((theta_x_sn[0] - lens_x)**2 + (theta_y_sn[0] - lens_y)**2)
            sep_sn1_lens = torch.sqrt((theta_x_sn[1] - lens_x)**2 + (theta_y_sn[1] - lens_y)**2)
            
            try:
                beta_x_host = self.src_host.x0.value.float()
                beta_y_host = self.src_host.y0.value.float()
                theta_x_host, theta_y_host = self.lens.forward_raytrace(beta_x_host, beta_y_host)
                sep_sn0_host = torch.min(torch.sqrt((theta_x_sn[0] - theta_x_host)**2 + (theta_y_sn[0] - theta_y_host)**2))
                sep_sn1_host = torch.min(torch.sqrt((theta_x_sn[1] - theta_x_host)**2 + (theta_y_sn[1] - theta_y_host)**2))
                separations = [sep_sn_sn, sep_sn0_host, sep_sn1_host, sep_sn0_lens, sep_sn1_lens]
            except (IndexError, ValueError):
                separations = [sep_sn_sn, sep_sn0_lens, sep_sn1_lens]
            
            max_sep = torch.max(torch.stack(separations))
            min_sep = torch.min(torch.stack(separations))

            # Magnification at each SN image position
            mag_sn0 = self.lens.magnification(theta_x_sn[0].unsqueeze(0), theta_y_sn[0].unsqueeze(0))
            mag_sn1 = self.lens.magnification(theta_x_sn[1].unsqueeze(0), theta_y_sn[1].unsqueeze(0))
            print(f"Magnification image 0: {mag_sn0.item():.4f}")
            print(f"Magnification image 1: {mag_sn1.item():.4f}")
            print(f"Relative magnification (image 0 / image 1): {(mag_sn0/mag_sn1).item():.4f}")
            print(f"Relative magnification (image 1 / image 0): {(mag_sn1/mag_sn0).item():.4f}")

            
            return sn_td_relative, n_images, max_sep, min_sep, sep_sn_sn, sep_sn0_lens, sep_sn1_lens
           
            # if 0.55 < min_sep < 0.65:  # adjust range as needed
            #     return sn_td_relative, n_images, min_sep
                
        return None



In [10]:
f = h5py.File('lsst-altsched-1a-lowz.h5','r')
simitems = np.asarray(f['system']['block0_items'][()],dtype=str)
simdataall = f['system']['block0_values'][()]

# indcatalog = range(len(simdataall))
# indcatalog = np.loadtxt("goldstein_inds_for_20_day_tds.txt", dtype=int).tolist()
# indcatalog = [1125, 1327, 1756, 5359, 9445,     
#               18526, 20673, 20999, 24621,  #19892
#               25728, 35825, 40581, 43887, 46513,
#               47818, 48914, 50502]

# indcatalog = [62425, 64780, 64868, 65870, 67031, 69457, 73735,
#              81728, 83442, 83673, 86666, 87725, 89052, 90478, 92211, 92766, 
#               94568, 95086, 95188, 95508, 99484, 105074, 106898, 107015, 107717, 
#               #112874, 123375, 126669, 131892, 132393, 134883, 141797, #113779  122345
#               145977, 147884, 148205, 149094, 149948, 150897, 151172, 152697, 153791,
#               155028, 158230, 162317, 168353, 170529, 170833, 173667, 174611, 174776,
#               180112, 180174, 181686, 183219, 185572, 192676, 193620, 193744, 194796
#              ]
indcatalog = [96679]#[77811, 2737, 208325, 150071, 13102, 63815, 115323, 170330] #[53838, 225080, 111827, 64783, 137171,98788, 50498, 167791, 220704, 180616, 51409, 152744, 175741, 157804, ]  #[77811, 2737, 208325, 150071, 13102, 63815, 115323, 170330]  #[62425, 216276, 209546, 105422]  


indcatalog = [62425, 216276, 209546, 111827, 64783, 157804, 2737, 170330, 50421]

for i, system_indx in enumerate(indcatalog):
    simdata=simdataall[system_indx]
    
    z_lens = simdata[simitems == "zl"][0]
    z_source = simdata[simitems == "zs"][0]
    # print('z lens',z_lens,'z source',z_source)
    if z_source < 1:
    
        sn_x = simdata[simitems == "snx"][0]
        sn_y = simdata[simitems == "sny"][0]
        
        host_reff = simdata[simitems == "host_reff"][0]      # units? assuming arcsec
        host_n = simdata[simitems == "host_n"][0]          
        host_x = simdata[simitems == "host_x"][0]            # units? assuming arsec
        host_y = simdata[simitems == "host_y"][0]            # units? assuming arcsec
        host_theta = simdata[simitems == "host_theta"][0]    # units? assuming radians
        host_ellip = simdata[simitems == "host_ellip"][0]
        #host_mag = ???? 
        sh = galsim.Shear(e=host_ellip, beta=host_theta*galsim.radians)
        host_e1 = sh.e1
        host_e2 = sh.e2
        
        lens_reff = simdata[simitems == "lensgal_reff"][0]    # units? assuming arcsec
        lens_n = simdata[simitems == "lensgal_n"][0]
        lens_x = simdata[simitems == "lensgal_x"][0]          # units? assuming arcsec
        lens_y = simdata[simitems == "lensgal_y"][0]          # units? assuming arcsec
        theta_E_sim = simdata[simitems == "theta_e"][0]       # units? idl what is going on here
        lens_theta = simdata[simitems == "lensgal_theta"][0]  # units? assuming degrees
        lens_ellip = simdata[simitems == "lensgal_ellip"][0]
        s = galsim.Shear(e=lens_ellip, beta=lens_theta*galsim.degrees)
        lens_e1 = s.e1
        lens_e2 = s.e2
        
        # Einstein radius calculation done by Charlotte in OG Lenstronomy code - just copied
        Dl = cosmo.angular_diameter_distance(simdata[simitems == "zl"][0]).value
        Ds = cosmo.angular_diameter_distance(simdata[simitems == "zs"][0]).value
        Dls = cosmo.angular_diameter_distance_z1z2(simdata[simitems == "zl"][0],simdata[simitems == "zs"][0]).value
        sig = simdata[simitems == "sigma"][0]
        theta_E = 4*np.pi*(sig/3e5)**2*Dls/Ds*180/np.pi*60*60    # units = arcsec??
        
        
        # call later as: flux = sn_lc_func_g(time_tensor):
        roman_seeing_g, roman_seeing_r, roman_seeing_i = telescope_params['Roman']['seeing'][roman_bands[0]], telescope_params['Roman']['seeing'][roman_bands[1]], telescope_params['Roman']['seeing'][roman_bands[2]]
        roman_mag_zero_point_g, roman_mag_zero_point_r, roman_mag_zero_point_i = telescope_params['Roman']['zero_points_mags'][roman_bands[0]], telescope_params['Roman']['zero_points_mags'][roman_bands[1]], telescope_params['Roman']['zero_points_mags'][roman_bands[2]]
        roman_sn_lc_func_g = make_caustics_lightcurve_func(t_min=0, t_max=150, band=roman_bands[0], zp=roman_mag_zero_point_g, z_source=z_source)
        roman_sn_lc_func_r = make_caustics_lightcurve_func(t_min=0, t_max=150, band=roman_bands[1], zp=roman_mag_zero_point_r, z_source=z_source)
        roman_sn_lc_func_i = make_caustics_lightcurve_func(t_min=0, t_max=150, band=roman_bands[2], zp=roman_mag_zero_point_i, z_source=z_source)
        
        
        
        # Cosmology model
        cosmology = caustics.FlatLambdaCDM(name="cosmo")
        
        host_Ie_dict = {"g": 0.3,    "r": 0.5,    "i": 1,
                        roman_bands[0]: 0.3, roman_bands[1]: 0.5, roman_bands[2]: 1}        # these are random numbers
        lens_Ie_dict = {"g": 0.1,    "r": 0.2,     "i": 0.3,
                        roman_bands[0]: 0.1, roman_bands[1]: 0.2, roman_bands[2]: 0.3}      # these are random numbers
        
        # Host galaxy light model
        host_light = caustics.Sersic(
            name="hostlight",
            x0 = host_x,              # The x-coordinate of the Sersic source's center [arcsec] - tensor
            y0 = host_y,              # The y-coordinate of the Sersic source's center [arcsec] - tensor
            # q=1 - 0.29,             # The axis ratio of the Sersic source [unitless] - tensor
            # phi=-30 * pi / 180,     # The orientation of the Sersic source (position angle) [radians] - tensor
            angle_system="e1_e2",     # change angle system since I have e from Goldstein and used e1, e2 in Lenstronomy??
            e1 = host_e1,             # e1, e2 calculated using Galsimn
            e2 = host_e2,
            n = host_n,               # The Sersic index [unnitless] - tensor
            Re =  host_reff,          # The scale length of the Sersic source [arcesc] - tensor
            Ie = host_Ie_dict["g"],   # The intensity at the effective radius [flux] - tensor
        )
        
        # Lens Light model
        lens_light = caustics.Sersic(
            name="lens",
            x0 = lens_x,
            y0 = lens_y,
            # q = 1 - 0.29,
            # phi = -30 * pi / 180,
            angle_system = "e1_e2",     
            e1 = lens_e1,            
            e2 = lens_e2,
            n = lens_n,
            Re = lens_reff,
            Ie = torch.tensor(lens_Ie_dict["g"]),
        )
        
        # Lens mass model
        lens_mass = caustics.SIE(
            name="sie",
            cosmology=cosmology,
            z_s = z_source,                                          # The redshift of the source.
            z_l = z_lens,                                            # The redshift of the lens.
            x0 = torch.tensor(lens_x, dtype=torch.float32),          # The x-coordinate of the lens center. [arcsec]
            y0 = torch.tensor(lens_y, dtype=torch.float32),          # The y-coordinate of the lens center.[arcsec]
            # q=1 / 1.14,
            # phi=pi / 2 + 1.6755160819145565,
            Rein = torch.tensor(theta_E, dtype=torch.float32),       # The Einstein radius of the lens. [arcsec]
            angle_system = "e1_e2", 
            e1 = lens_e1,            
            e2 = lens_e2,
        )
        
        # SN light model
        sn_light = TimeVaryingGaussian(
            name="SN",
            x0 = torch.tensor(sn_x, dtype=torch.float32),              
            y0 = torch.tensor(sn_y, dtype=torch.float32),
            lc_funcs={roman_bands[0]: roman_sn_lc_func_g, roman_bands[1]: roman_sn_lc_func_r, roman_bands[2]: roman_sn_lc_func_i}
        )
        
        
        # We are simulating (not fitting) — freeze all parameters
        host_light.to_static(local_only=False)
        lens_light.to_static(local_only=False)
        sn_light.to_static(local_only=False)
        
        
        # Roman PSF and image resolution
        roman_psf_dict = {}
        if system_indx == 2737 or system_indx == 170330 or system_indx == 50421:
            size = 10.0  # width of the image in arc seconds  --> old Lenstronomy code
        else:
            size = 6.0
        
        
        roman_pixel_scale = telescope_params['Roman']['pixel_scale']
        roman_numpix = int(round(size / roman_pixel_scale))
        
        # Instantiate simulator
        roman_simulator = Singlelens(
            lens_mass=lens_mass,
            lens_light=lens_light,
            host_light=host_light,
            sn_light = sn_light,
            band_keys=roman_bands, 
            host_Ie_dict = host_Ie_dict,
            lens_Ie_dict = lens_Ie_dict,
            pixels_x = roman_numpix,
            pixelscale = roman_pixel_scale,
            upsample_factor=5,
            psf_dict=roman_psf_dict,
        )
        
        # Set all parameters to be dynamic
        roman_simulator.to_dynamic(local_only=False)
        cosmology.to_static()  # except cosmology parameters
        
        # Now create a flattened tensor
        roman_params_for_simulator = roman_simulator.get_values()
        # print("Roman Params tensor shape: ", roman_params_for_simulator.shape)
        roman_params_for_simulator = roman_simulator.get_values().float()   # Force all parameters to float32 (got mismatch error earlier)
        
        # obs_times = torch.arange(0, 150, 2, dtype=torch.float32)
        obs_times = torch.tensor([50], dtype=torch.float32)
        
        result = roman_simulator.run_simulator(roman_params_for_simulator, times=obs_times, psf_dict=roman_psf_dict)
    
        if result is not None:
            sn_td_relative, n_images, max_sep, min_sep, sep_sn_sn, sep_sn0_lens, sep_sn1_lens = result
            # if max_sep < 0.6:
                
            print('Index: ', system_indx)
            print(f"Delay: {sn_td_relative:.2f} days")
            print(f" Max Sep: {max_sep:.2f}")
            print(f" Min Sep: {min_sep:.2f}")
            print(f" SN-SN Sep: {sep_sn_sn:.2f}")
            print(f" SN1-lens Sep: {sep_sn0_lens:.2f}")
            print(f" SN2-lens Sep: {sep_sn1_lens:.2f}")
            print('N delays: ', n_images)
            print('z lens',z_lens,'z source',z_source)
            print('\n')
        else:
            # Optional: uncomment to see progress, but it will be a lot of text!
            # print(f"Index {system_indx}: Did not fit criteria.")
            continue
    else:
        continue





Magnification image 0: 1.7580
Magnification image 1: 0.5015
Relative magnification (image 0 / image 1): 3.5057
Relative magnification (image 1 / image 0): 0.2853
Index:  62425
Delay: 13.61 days
 Max Sep: 0.59
 Min Sep: 0.11
 SN-SN Sep: 0.58
 SN1-lens Sep: 0.49
 SN2-lens Sep: 0.11
N delays:  2
z lens 0.46161739165701493 z source 0.9253593659777268


Magnification image 0: 1.4457
Magnification image 1: 1.7465
Relative magnification (image 0 / image 1): 0.8278
Relative magnification (image 1 / image 0): 1.2080
Index:  216276
Delay: 13.51 days
 Max Sep: 0.57
 Min Sep: 0.10
 SN-SN Sep: 0.57
 SN1-lens Sep: 0.12
 SN2-lens Sep: 0.45
N delays:  2
z lens 0.5126401146679617 z source 0.9339237713042305


Magnification image 0: 2.1329
Magnification image 1: 2.1314
Relative magnification (image 0 / image 1): 1.0007
Relative magnification (image 1 / image 0): 0.9993
Index:  209546
Delay: 10.87 days
 Max Sep: 0.60
 Min Sep: 0.16
 SN-SN Sep: 0.60
 SN1-lens Sep: 0.44
 SN2-lens Sep: 0.16
N delays:  2
z l